# Introduction to atomman: Polycrystal system construction

__Lucas M. Hale__, [lucas.hale@nist.gov](mailto:lucas.hale@nist.gov?Subject=ipr-demo), _Materials Science and Engineering Division, NIST_.
    
[Disclaimers](http://www.nist.gov/public_affairs/disclaimer.cfm) 

## 1. Introduction <a id='section1'></a>

This Notebook provides usage details and examples for generating polycrystalline systems using Voronoi tesselation.

**Library Imports**

In [3]:
# Standard Python libraries
import datetime

# http://www.numpy.org/
import numpy as np

# https://github.com/usnistgov/atomman
import atomman as am

# Show atomman version
print('atomman version =', am.__version__)

# Show date of Notebook execution
print('Notebook executed on', datetime.date.today())

/home/lmh1/conda/envs/ipr/lib/python3.13/site-packages/numpy/_core/getlimits.py:552: UserWarning: Signature b'\x00\xd0\xcc\xcc\xcc\xcc\xcc\xcc\xfb\xbf\x00\x00\x00\x00\x00\x00' for <class 'numpy.longdouble'> does not match any known type: falling back to type probe function.
This warnings indicates broken support for the dtype!
  machar = _get_machar(dtype)


atomman version = 1.5.2
Notebook executed on 2026-03-26


## 2. Method and Theory

**Caution**: As this method generates the polycrystal grains purely from geometric considerations, the resulting grain boundaries may not be realistic! Always check the final system and remove atoms that are too close together, and anneal/relax the structure!

One of the classic ways of generating polycrystal atomic configurations is using Voronoi tesselation to divide the full cell space into distinct regions, then filling in those regions with atoms associated with crystals in different orientations.  The atomman.defect.Polycrystal class manages such atomic system creation.

The method used by Polycrystal follows the following steps

1. A cell Box and number of grains are defined.  Each grain is assigned a point and a rotation matrix, which can be manually specified or randomly selected.
2. The points are passed to a Voronoi solver, which returns Voronoi vertices and additional information.
3. The Voronoi solution is used to generate a PlaneSet region object for each grain.
4. Each grain region is filled with atoms based on a reference unit cell and the grain's rotation matrix. This is accomplished by creating a supercell of the unit cell, rotating and shifting it to overlap the grain region, then extracting only the atoms that are inside the grain's region.
6. A final system object is generated by combining the cell Box with the atoms from each grain.

The Polycrystal class currently supports two Voronoi solver methods:
- 'freud' uses the freud.locality.Voronoi class. This method requires installing the optional freud package (pip install freud-analysis).
- 'scipy' uses the scipy.spatial.Voronoi class.
  
The two methods give similar but not identical final structures. This is due to scipy/numpy using double-precision floating point numbers while freud uses single-precision. Also, freud tends to be faster as its Voronoi solver inherently allows for periodic boundaries whereas scipy's requires constructing a supercell. The difference in speed is small unless you are generating a large number of grains.


## 3. Example polycrystal generation

In [4]:
# Define the cell box for the final configuration
box = am.Box(lx=200, ly=200, lz=200)

# Define the reference crystal unit cell for populating the grains
ucell = am.System(atoms=am.Atoms(pos=[[0.0, 0.0, 0.0], [0.5, 0.5, 0.5]]), box=am.Box.cubic(3.665), symbols='Fe', scale=True)

### 3.1 Polycrystal initialization

Initializing a Polycrystal object defines the number grains, the grain points and rotations.  If method != 'no', it will also call one of the Voronoi method solvers to identify the regions associated with each grain.

- __box__ (*atomman.Box*) The cell box of the final atomic configuration to be generated.
- __ucell__ (*atomman.System*) The reference crystal unit cell to use for filling in the grain regions.
- __ngrains__ (*int or None, optional*) The number of grains to create.  Optional if points and/or rotations are given.
- __points__ (*array-like or None, optional*) An (Nx3) array-like object where N = ngrains that specifies the seed points used for the Voronoi analysis.  If not given, random points will be selected.
- __rotations__ (*array-like or None, optional*) An (Nx3x3) array-like object where N = ngrains that specifies the rotation transformation matrices to apply to ucell's positions for a given grain.  If not given, random rotations will be selected using an algorithm that uniformly samples the spherical rotation space.
- __method__ (*str or None, optional*) Specifies the method for the Voronoi calculation and grain region identification.  'freud' uses the freud package, 'scipy' uses the scipy package, and 'no' will only define the points and not solve. Default value of None uses 'freud' if the package is installed, and uses 'scipy' otherwise.
- __rng__ (*numpy.random.Generator or None, optional*) An existing random number generator to use.  If None (default), a new generator will be created based on seed.
- __seed__ (*int or None, optional*) A random number seed to use when creating a generator if rng is not given.  If None (default), then then fresh, unpredictable entropy will be pulled from the OS.
- __scale__ (*bool, optional*) Indicates if points (if given) are to be taken as Cartesian (False, default) or relative/fractional (True) with respect to box.

In [29]:
polycrystal = am.defect.Polycrystal(box, ucell, ngrains=10)

### 3.2. build_polycrystal()

Once the grains have been defined and regions solved, you can call build_polycrystal() to generate an atomic configuration.  

- __ucell__ (*atomman.System or None, optional*) Specifies the crystal unit cell to use for generating the grains. If None (default), will use the ucell set during class initialization.
- __sizemults__ (*tuple or None, optional*) System.supersize() size multipliers to use on ucell.  If given, the resulting system should be large enough that the grain region fits inside the supercell for any rotation.  Default value of None will guess sizemults for you.
- __verbose__ (*bool*) If set to True and the freud method was used, will print out the number of atoms in each grain compared to the estimated expected number based on ucell's density and the grain's volume.


In [6]:
system = polycrystal.build_polycrystal()

In [7]:
print(system.natoms)

324986


Example polycrystal structure image (atoms colored by polyhedral template mapping)

![alt text](files/polycrystal_example.png)

---

## 4. Component functions

These are functions that are or may be called during Polycrystal initialization that can be directly accessed allowing users more control over the operations.

### 4.1. create_grains()

Calling the create_grains() method will generates new grains based on new grain points and rotations. 
- The method takes the same parameters as the class init, excluding ucell. However, all parameters are now optional.
- If ngrains, points and rotations are all not specified, it will pick new random points and rotations for the same number of grains as were previously defined.

### 4.2. solve_grains_freud() and solve_grains_scipy()

These are the underlying Voronoi solution methods that find the Voronoi vertices and build the grain regions. Calling them directly will overwrite the grain region definitions already set. As they do not change the grain points and rotations, you could call them individually to compare the structures generated by the two methods.

## 5. Grain objects

The Polycrystal class defines the grains using a list of Grain objects.

### 5.1. Grain attributes

Each grain tracks the following grain properties
- __point__ (*numpy.ndarray*) The voronoi seed/reference position for the grain.    
- __rotation__ (*numpy.ndarray*) The rotation matrix for orienting the grain.
- __region__ (*atomman.region.PlaneSet or None*) Region object used for atom selection.
- __volume__ (*float or None*) The volume of the region (only computed with freud).
- __extra_data__ (*dict*) extra method-specific data fields that could be used for verification checks.



In [10]:
grain = polycrystal.grains[0]
print(grain.point)
print(grain.rotation)
print(grain.volume)

[ 44.17354002 132.06589345  88.845091  ]
[[ 0.67263441  0.72220158 -0.16120742]
 [-0.11684057 -0.11146569 -0.98687572]
 [-0.7306923   0.68264213  0.00940678]]
334976.2494482474


In [12]:
print(grain.extra_data.keys())

dict_keys(['vertices', 'freud_vertices', 'neighbor_points'])


### 5.2. Grain methods

Each grain tracks the following grain properties
- __solve_region_freud()__ used by Polycrystal.solve_grains_freud() to find the grain's region based on freud Voronoi outputs.
- __solve_region_scipy()__ used by Polycrystal.solve_grains_scipy() to find the grain's region based on scipy Voronoi outputs.
- __build_atoms()__ used by Polycrystal.build_polycrystal() to fill in atoms within the grain's region.
